# nanoasr (JAX): Train a Conformer-CTC on TPU

Train a Conformer-CTC model on LibriSpeech using **JAX + Flax NNX + optax**.

**Runtime**: Select a **TPU** runtime for maximum speed (`Runtime > Change runtime type > v5e-1 TPU` or `v6e-1 TPU`).  Any GPU runtime works too.

In [ ]:
!pip install -q flax optax librosa soundfile
!pip install -q git+https://github.com/Vaibhavdixit02/nanoasr.git

In [ ]:
import jax
import jax.numpy as jnp

devices = jax.devices()
print(f"JAX backend : {jax.default_backend()}")
print(f"Devices ({len(devices)}): {devices}")

## 1. Inspect the data pipeline

Load a few LibriSpeech samples and visualise the mel spectrogram.

In [ ]:
import os, soundfile as sf, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from nanoasr.jax.data import LibriSpeechDataset
from nanoasr.vocab import decode_indices

os.makedirs("./data", exist_ok=True)
ds = LibriSpeechDataset(root="./data", split="dev-clean")
print(f"{len(ds)} utterances")

In [ ]:
mel, tokens = ds[0]
text = decode_indices(tokens.tolist())
print(f"Text: {text}")
print(f"Mel shape: {mel.shape}  (n_mels, T)")
print(f"Token length: {len(tokens)}")

audio_path = ds.samples[0][0]
waveform, sr = sf.read(audio_path)
display(Audio(waveform, rate=sr))

fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(mel, aspect="auto", origin="lower")
ax.set_xlabel("Time frames")
ax.set_ylabel("Mel bins")
ax.set_title(f'"{text}"')
plt.tight_layout()
plt.show()

## 2. Model

The `depth` parameter controls everything: `d_model = depth * 32`, `n_heads = depth`, `n_layers = depth`.

| depth | d_model | heads | layers | ~params |
|-------|---------|-------|--------|---------|
| 4     | 128     | 4     | 4      | 2 M     |
| 8     | 256     | 8     | 8      | 10 M    |
| 12    | 384     | 12    | 12     | 30 M    |

In [ ]:
import flax.nnx as nnx
from nanoasr.jax.model import Conformer, get_config

depth = 8
config = get_config(depth)
model = Conformer(config, rngs=nnx.Rngs(params=0, dropout=1))

n_params = sum(x.size for x in jax.tree.leaves(nnx.state(model, nnx.Param)))
print(config)
print(f"{n_params:,} parameters")

## 3. Train

Train on `train-clean-100` (~28 k utterances, ~100 hours) and evaluate WER on `dev-clean` every 5 epochs.

Checkpoints save to Google Drive every epoch so you never lose progress if Colab disconnects.  To resume a crashed run, set `resume=` to the last checkpoint path.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/nanoasr-jax"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
from nanoasr.jax.train import train

model, config = train(
    depth=8,
    data="train-clean-100",
    eval_data="dev-clean",
    data_root="./data",
    epochs=100,
    batch_size=64,
    eval_every=5,
    save_dir=save_dir,
    # resume=f"{save_dir}/model_depth8_last.pkl",  # uncomment to resume
)

## 4. Evaluate best checkpoint

Load the best checkpoint from Drive and run full eval on dev-clean.

In [ ]:
from nanoasr.jax.eval import evaluate_checkpoint

results = evaluate_checkpoint(
    f"{save_dir}/model_depth{depth}_best.pkl",
    eval_split="dev-clean",
    data_root="./data",
)